# 19. SOTA Execution: Dense CWT Deep Vision & Hybrid Dictionary
**Objective:** Use Disk-Backed HDF5 streaming to train a 2D CWT CNN on massive dense data without crashing system RAM, then extract hybrid dictionary features.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import time
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR, DEVICE
from src.cwt_features import extract_cwt_spectrograms
from src.cwt_net import CWTVisionNet

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load the Dense Dataset Indexes

In [2]:
dense_h5_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1_Dense.h5")
cwt_h5_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1_CWT_Dense.h5")

with h5py.File(dense_h5_path, 'r') as f:
    y_dense = np.array(f['y']).astype(np.int64)
    reps_dense = np.array(f['reps'])
    total_windows = f['X'].shape[0]

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_dense, train_reps))[0]
test_idx = np.where(np.isin(reps_dense, test_reps))[0]

print(f"Total Dense Windows: {total_windows}")

Total Dense Windows: 186753


### 2. Memory-Safe Batched CWT Extraction
Extracts spectrograms in chunks and streams them directly to the hard drive.

In [3]:
if not os.path.exists(cwt_h5_path):
    print(f"Creating memory-mapped HDF5 file at {cwt_h5_path}...")
    
    with h5py.File(dense_h5_path, 'r') as f_in, h5py.File(cwt_h5_path, 'w') as f_out:
        # Create empty dataset on disk for the CWT images
        f_out.create_dataset('X_cwt', shape=(total_windows, 16, 20, 10), dtype=np.float32, chunks=True)
        f_out.create_dataset('y', data=y_dense)
        f_out.create_dataset('reps', data=reps_dense)
        
        batch_size = 5000  # Safe batch size for RAM
        start_time = time.time()
        
        for i in range(0, total_windows, batch_size):
            end = min(i + batch_size, total_windows)
            print(f"Extracting CWT for windows {i} to {end}...")
            
            # Extract just this batch
            batch_raw = np.array(f_in['X'][i:end])
            batch_cwt = extract_cwt_spectrograms(batch_raw, n_scales=16, n_jobs=-1)
            
            # Save directly to disk as float32
            f_out['X_cwt'][i:end] = np.float32(batch_cwt)
            
            # Force RAM clear
            del batch_raw, batch_cwt
            gc.collect()
            
        elapsed = time.time() - start_time
        print(f"Disk-backed CWT Extraction completed in {elapsed / 60:.2f} minutes.")
else:
    print(f"HDF5 file {cwt_h5_path} already exists. Skipping extraction.")

HDF5 file /workspaces/TCC/data/preprocessed/DB1_subject_1_CWT_Dense.h5 already exists. Skipping extraction.


### 3. High-Speed RAM-Safe PyTorch Dataset

In [4]:
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score

print("Loading full CWT tensor into RAM cleanly (~2.4 GB)...")
with h5py.File(cwt_h5_path, 'r') as f:
    # Load the entire float32 dataset into memory exactly once
    X_all_cwt = f['X_cwt'][:]
    y_all = f['y'][:]

class RAMFastDataset(Dataset):
    """
    Holds exactly ONE copy of the array in RAM. 
    Uses indices to fetch data, preventing OOM crashes from array duplication.
    """
    def __init__(self, X_full, y_full, indices):
        self.X_full = X_full
        self.y_full = y_full
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x_data = self.X_full[real_idx]
        y_data = self.y_full[real_idx]
        
        # Transpose: (Freq, Time, Channels) -> (Channels, Freq, Time)
        x_tensor = torch.tensor(x_data, dtype=torch.float32).permute(2, 0, 1)
        y_tensor = torch.tensor(y_data, dtype=torch.long)
        return x_tensor, y_tensor

# The DataLoaders now read from RAM instantly without copying the massive array
train_loader = DataLoader(RAMFastDataset(X_all_cwt, y_all, train_idx), batch_size=256, shuffle=True)
test_loader = DataLoader(RAMFastDataset(X_all_cwt, y_all, test_idx), batch_size=256, shuffle=False)
print("DataLoaders perfectly configured for high-speed training.")

Loading full CWT tensor into RAM cleanly (~2.4 GB)...
DataLoaders perfectly configured for high-speed training.


### 4. Train the Deep Vision CNN at Full Speed

In [5]:
num_classes = int(max(y_dense.max(), y_dense.max()) + 1)
cnn_model = CWTVisionNet(num_classes).to(DEVICE)

# Calculate class weights for imbalance
y_train_subset = y_all[train_idx]
weights = np.ones(num_classes, dtype=np.float32)
classes, counts = np.unique(y_train_subset, return_counts=True)
weights[classes] = 1.0 / counts
tensor_weights = torch.FloatTensor(weights / weights.sum()).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=tensor_weights)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

epochs = 30 
print(f"\nTraining CWT Deep Vision Network on {DEVICE}...")
for epoch in range(epochs):
    cnn_model.train()
    total_loss = 0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        out = cnn_model(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

# Evaluate CNN Baseline
cnn_model.eval()
preds = []
with torch.no_grad():
    for bx, _ in test_loader:
        out = cnn_model(bx.to(DEVICE))
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())

cnn_acc = accuracy_score(y_all[test_idx], preds)
print(f"\n--- SOTA Dense CNN Performance ---")
print(f"Accuracy: {cnn_acc * 100:.2f}%")


Training CWT Deep Vision Network on cuda...
Epoch 1/30 | Loss: 1.9942
Epoch 2/30 | Loss: 1.4144
Epoch 3/30 | Loss: 1.2485
Epoch 4/30 | Loss: 1.1498
Epoch 5/30 | Loss: 1.0719
Epoch 6/30 | Loss: 1.0140
Epoch 7/30 | Loss: 0.9688
Epoch 8/30 | Loss: 0.9301
Epoch 9/30 | Loss: 0.8934
Epoch 10/30 | Loss: 0.8637
Epoch 11/30 | Loss: 0.8308
Epoch 12/30 | Loss: 0.8126
Epoch 13/30 | Loss: 0.7877
Epoch 14/30 | Loss: 0.7689
Epoch 15/30 | Loss: 0.7477
Epoch 16/30 | Loss: 0.7279
Epoch 17/30 | Loss: 0.7098
Epoch 18/30 | Loss: 0.6926
Epoch 19/30 | Loss: 0.6815
Epoch 20/30 | Loss: 0.6631
Epoch 21/30 | Loss: 0.6514
Epoch 22/30 | Loss: 0.6386
Epoch 23/30 | Loss: 0.6296
Epoch 24/30 | Loss: 0.6147
Epoch 25/30 | Loss: 0.6105
Epoch 26/30 | Loss: 0.5982
Epoch 27/30 | Loss: 0.5858
Epoch 28/30 | Loss: 0.5752
Epoch 29/30 | Loss: 0.5648
Epoch 30/30 | Loss: 0.5589

--- SOTA Dense CNN Performance ---
Accuracy: 56.05%


### 5. The Hybrid Upgrade: Dictionary Learning on Deep Features

In [7]:
feature_extractor = cnn_model.features

def extract_deep_features(loader, model_features):
    features_list = []
    with torch.no_grad():
        for bx, _ in loader:
            feat = model_features(bx.to(DEVICE))
            features_list.append(feat.view(feat.size(0), -1).cpu().numpy())
    return np.vstack(features_list)

print("Extracting 128D Deep Visual Features...")
X_train_deep = extract_deep_features(train_loader, feature_extractor)
X_test_deep = extract_deep_features(test_loader, feature_extractor)

print(f"Original Deep Features Shape: {X_train_deep.shape}")

# FIX: Reshape 2D (Batch, 128) -> 3D (Batch, 128, 1 Channel)
X_train_deep_3d = X_train_deep.reshape(-1, 128, 1)
X_test_deep_3d = X_test_deep.reshape(-1, 128, 1)

from src.dictionary_freq import FrequencyDictionaryLearner, FrequencyOMPExtractor

# We use 64 atoms to compress the 128-dimensional deep feature space
hybrid_learner = FrequencyDictionaryLearner(n_atoms=64, transform_n_nonzero_coefs=5, method='minibatch')
print("\nTraining Hybrid Dictionary...")
hybrid_learner.fit(X_train_deep_3d)

hybrid_extractor = FrequencyOMPExtractor(hybrid_learner.dictionary_, n_nonzero_coefs=5)
X_train_hybrid_sparse = hybrid_extractor.transform(X_train_deep_3d)
X_test_hybrid_sparse = hybrid_extractor.transform(X_test_deep_3d)

print(f"Hybrid Sparse Code Shape: {X_train_hybrid_sparse.shape}")

Extracting 128D Deep Visual Features...
Original Deep Features Shape: (126940, 128)

Training Hybrid Dictionary...
Training MINIBATCH Dictionary with 64 atoms on 126940 frequency spectra...
Dictionary learning complete.
Hybrid Sparse Code Shape: (126940, 64)


### 6. Final Evaluation of Hybrid Features

In [8]:
from sklearn.preprocessing import StandardScaler
from src.classification import train_svm_classifier, evaluate_classifier

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hybrid_sparse)
X_test_scaled = scaler.transform(X_test_hybrid_sparse)

print("\n--- Training SVM on Hybrid CWT + Dictionary Features ---")
# Using Linear Kernel for high-dimensional sparse data speed
hybrid_svm = train_svm_classifier(X_train_scaled, y_all[train_idx], kernel="linear", C=1.0)
results = evaluate_classifier(hybrid_svm, X_test_scaled, y_all[test_idx])

print(f"\nHybrid Dictionary + SVM Accuracy: {results['accuracy'] * 100:.2f}%")


--- Training SVM on Hybrid CWT + Dictionary Features ---
Training SVM (Kernel: linear) on 126940 samples...
Training complete.
Evaluating model on test set...

Hybrid Dictionary + SVM Accuracy: 1.26%
